Tenemos 2 DEMs que al verlos en un visor GIS esta uno desplazado respecto del otro en la vertical  

# CORREGIR VERTICALMENTE LOS DEMS
(entre ellos => modificaras uno solo!!!)  
Si tenes altura de referencia, en el código 03 dejaré como mover ambos DEMs (inicial y final) a H de ref.

Lo primero que harás es crear en un visor GIS un shapefile con poligonos a lo largo de tu imagen que luego los usaras para determinar la altura de cada DEM en esos poligonos_afloramientos y así poder comprarlas y mover un DEM hacia el otro.  
**(TOMA AFLORAMIENTOS PREFERENTEMENTE PLANOS Y EN BAJA ELEVACIÓN)**  
Tené en cuenta que tu shape tiene que tener los poligono enumerados por valores unicos en una **columna** llamada **'ID'** para que este codigo corra!

OJO, buscar las indicaciones ***# MODIFICAR*** porque son las variables y datos que **tiene que cambiar el usuario del código**

In [ ]:
from termcolor import colored

import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal
import os

from matplotlib import pyplot as plt
import matplotlib.image as mpimg

import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.plot import show_hist

#import funciones as fn

import sys
sys.path.append('../')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def plote_porcent(banda_del_arreglo, banda = 'banda', p = 0, nodata = None, figsize = (12,6)): 
    maxim = 100 -p
    
    band = banda_del_arreglo
    
    min_b, max_b = np.min(band), np.max(band)
    p_b_min = np.percentile(band, p)
    p_b_max = np.percentile(band, 100-p)
     
    plt.figure(figsize = figsize)
    plt.hist(band.ravel(), bins = 100)
    plt.axvline(p_b_min, color = 'red' ,linestyle = '--', label = f'Percentil {p} %')
    plt.axvline(p_b_max, color = 'black' ,linestyle = '--', label = f'Percenil {maxim}%')
    plt.legend()
    plt.title(f'Histograma {banda}')
    #Como hago para que el titulo me tome 
    plt.show()

In [ ]:
def porcentajes(banda_del_arreglo, p = 0, nodata = None):    
    banda = banda_del_arreglo
    
    #Mínimo y máximo:
    min_b, max_b = np.min(banda), np.max(banda)

    #Percentiles 2% y 98%
    p_b_min = np.percentile(banda, p)
    p_b_max = np.percentile(banda, 100-p)

    print(f'Mínimo:{min_b}, Máximo:{max_b}')
    print(f'Percentil {p}%: {p_b_min}, Percentil {p}%: {p_b_max}')

In [ ]:
# MODIFICAR direcciorios, segun cual elijas mover (o los 2) (*)

dir_DEM_o = 'C:/' # MODIFICAR

dir_DEM_proc = 'C:/DEM_procesando' # MODIFICAR

DEM_i = 'tu_DEM_fechaInicial' # MODIFICAR
DEM_f = 'tu_DEM_fechaFinal' # MODIFICAR

archivo_H_ref = f'{dir_DEM_proc}/shapes/poligonos_afloramientos.shp' # MODIFICAR

archivo_DEM_i = f'{dir_DEM_o}/{DEM_i}.tif'
archivo_DEM_f = f'{dir_DEM_o}/{DEM_f}.tif'

archivo_DEM_i_2 = f'{dir_DEM_proc}/{DEM_i}_movido.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_i_2b = f'{dir_DEM_proc}/{DEM_i}_movido_nan.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2 = f'{dir_DEM_proc}/{DEM_f}_movido.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2b = f'{dir_DEM_proc}/{DEM_f}_movido_nan.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal

archivo_DEM_i_3 = f'{dir_DEM_proc}/{DEM_i}_igualadoH.tif'
archivo_DEM_f_3 = f'{dir_DEM_proc}/{DEM_f}_igualadoH.tif'
# uno de los dos (i o f) no se hará
# en este ejemplo vamos a mover f hacia i

Te recomiendo que vayas indentando con # los archivos que no corregis/moves/usas

OJO, que **tu shape tenga** los poligono enumerados por valores unicos en una columna llamada **'ID'** para que este codigo corra!

### PARA UN POLIGONO:

### SI TENES MAS DE UN POLÍGONO

Podes hacer directamente este codigo porque aúnque tu shape tenga un solo polígono SIRVE!

In [ ]:
# PARA EL DEM final

print('Valores de DEM final')
gdf = gpd.read_file(archivo_H_ref)

with rasterio.open(archivo_DEM_f_2) as src: # yo uso este porque moví el f respecto al i en la horizontal (ver código 01)
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)
    
    array_names = []
    medias_arrays = {}
    
    for idx, row in gdf.iterrows():
        geom = [row['geometry']]
        id_val = row['ID']
        
        out_image, _ = mask(src, geom, crop=True)
        out_image = out_image.astype(float)
        out_image[out_image == -9999] = np.nan
        out_image[out_image == 0] = np.nan
        
        var_array = f'array_f_ID_{id_val}'
        globals()[var_array] = out_image
        array_names.append(var_array)
        
        # Calcular media
        media_val = np.nanmean(out_image)
        
        var_media = f'media_array_f_ID_{id_val}'
        globals()[var_media] = media_val
        
        # También guardo en diccionario (opcional)
        medias_arrays[var_media] = media_val

print("✅ Arrays numpy creados:")
for name in array_names:
    print(f'📌 {name}')

print("\n✅ Variables con medias creadas:")
for name, valor in medias_arrays.items():
    print(f'📌 {name} = {valor:.4f}')

In [ ]:
# Plotiemos uno de ejemplo

if array_f_ID_1.ndim > 1: # Si tu array tiene varias bandas, tomá la primera:
    imagen = array_f_ID_1[0]  # primera banda
else:
    imagen = array_f_ID_1
plt.figure(figsize=(8,6))
plt.imshow(imagen, cmap='terrain')  # ó 'cmap' ó 'viridis'
plt.colorbar(label='Elevación')
plt.title('Recorte raster: array_ID_1')
plt.axis('off')
plt.show()

histo_ejemplo = plote_porcent(array_f_ID_1, banda = 'Dif', p = 5, nodata = None, figsize = (12,6))

In [ ]:
# PARA EL DEM inicial

print('Valores de DEM inicial')
gdf = gpd.read_file(archivo_H_ref)

with rasterio.open(archivo_DEM_i) as src: # Lo uso de referencia (ojo, mira en el visor GIS como estan tus imagenes)
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)
    
    array_names = []
    medias_arrays = {}
    
    for idx, row in gdf.iterrows():
        geom = [row['geometry']]
        id_val = row['ID']
        
        out_image, _ = mask(src, geom, crop=True)
        out_image = out_image.astype(float)
        out_image[out_image == -9999] = np.nan
        out_image[out_image == 0] = np.nan
        
        var_array = f'array_i_ID_{id_val}'
        globals()[var_array] = out_image
        array_names.append(var_array)
        
        # Calcular media
        media_val = np.nanmean(out_image)
        
        var_media = f'media_array_i_ID_{id_val}'
        globals()[var_media] = media_val
        
        # También guardo en diccionario (opcional)
        medias_arrays[var_media] = media_val

print("✅ Arrays numpy creados:")
for name in array_names:
    print(f'📌 {name}')

print("\n✅ Variables con medias creadas:")
for name, valor in medias_arrays.items():
    print(f'📌 {name} = {valor:.4f}')


In [ ]:
# TOMAR LA DIFERENCIA ENTRE AMBOS DEMs EN AFLORAMIENTOS DE ZONAS PLANAS (f-i)

dif_names = []
for id_val in gdf['ID']:
    var_f = f'media_array_f_ID_{id_val}'
    var_i = f'media_array_i_ID_{id_val}'
    var_dif = f'dif_ID_{id_val}'
    # Acceder a las variables con globals()
    media_f = globals().get(var_f)  
    media_i = globals().get(var_i)
    if media_i is not None and media_f is not None:
        globals()[var_dif] = media_f - media_i
        dif_names.append(var_dif)
    else:
        print(f"⚠️  Faltan datos para ID {id_val} (inicial o final)")
print("\nDiferencias entre DEMs:")
for name in dif_names:
    print(f'📌 {name} = {globals()[name]}')
    
# Recolectar todas las variables dif_ID_{id_val}
lista_diferencias = []
for id_val in gdf['ID']:
    var_dif = f'dif_ID_{id_val}'
    valor_dif = globals().get(var_dif)
    if valor_dif is not None:
        lista_diferencias.append(valor_dif)
    else:
        print(f"⚠️  No se encontró {var_dif}")
# Calcular la media general
media_diferencias = np.mean(lista_diferencias)
print(f"\n📊 Media de todas las diferencias: {media_diferencias}")
# También podés guardar la variable
globals()['media_diferencias_todos'] = media_diferencias

In [ ]:
aca deberías ver tu resultado y analizar si tiene lógica

### Mover verticalmente UNO de los DEMs

Si i fue mas alta que f debemos bajar 'archivo_DEM_i' en cantidad de  'media_diferencias'  
Si i fue mas baja que f debemos subir 'archivo_DEM_i' en cantidad de 'media_diferencias'  
...Hay que hacer **archivo_DEM_i + media_diferencias** (caso ejemplo me dió neg. va a bajarla)  
-------------------------------------------------------------------------------------------
Si f fue mas alta que i debemos bajar 'archivo_DEM_f' en cantidad de  'media_diferencias'  
Si f fue mas baja que i debemos subir 'archivo_DEM_i' en cantidad de 'media_diferencias'  
...Hay que hacer **archivo_DEM_f - media_diferencias** (caso ejemplo me dió neg. va a subirla)

TENÉ EN CUENTA QUE en este ejemplo uso _f , pero podes modificar los archivos en donde se marca *# MODIFICAR* y usar el _i

In [ ]:
DEM = gdal.Open(archivo_DEM_f_2b) # MODIFICAR si queres usar otro, yo uso el movido horizontalmente y en donde los valores sin dato son NaN
gt = DEM.GetGeoTransform()
src = DEM.GetProjection()
array_DEM = DEM.ReadAsArray()

In [ ]:
movida_verticalmente_f = array_DEM - media_diferencias
# ó podes usar un dif_ID_X
movida_verticalmente_i = array_DEM + media_diferencias

### Exportar DEM 
corregido verticalmente al otro DEM.

In [ ]:
filas = array_DEM.shape[0]
columnas = array_DEM.shape[1]
bandas = 1
driver = gdal.GetDriverByName('GTiff')
dem_salida = driver.Create(archivo_DEM_f_3, columnas, filas, bandas, gdal.GDT_Float32) # MODIFICAR si usaste otro, no es la idea dado que solo movemos f hacia i, pero tu camino podría ser el inverso y mover el i al f.
dem_salida.SetProjection(src)
dem_salida.SetGeoTransform(gt)
dem_salida.GetGeoTransform()
dem_salida.GetRasterBand(1).WriteArray(movida_verticalmente_f[:]) # ó MODIFICAR por movida_verticalmente_f
del dem_salida